In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from CBFV import composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import re
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor


/Users/christiannorman/py_envs/CBFV_env/lib/python3.12/site-packages/CBFV/composition.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:

class FlexibleNN(nn.Module):
    """
    A flexible neural network with configurable architecture for hyperparameter optimization.
    
    Parameters:
    -----------
    input_dim : int
        Number of input features (CBFV features)
    output_dim : int
        Number of output targets (phase fractions at selected temperatures)
    hidden_layers : list of int
        List of hidden layer sizes, e.g., [256, 128, 64]
    dropout_rate : float
        Dropout probability (0.0 to 1.0)
    dropout_type : str
        Type of dropout: 'standard', 'alpha' (for SELU), or 'none'
    activation : str
        Activation function: 'relu', 'leaky_relu', 'elu', 'selu', 'gelu', 'tanh'
    use_batch_norm : bool
        Whether to use batch normalization
    use_layer_norm : bool
        Whether to use layer normalization (alternative to batch norm)
    weight_decay : float
        L2 regularization strength (applied in optimizer, stored here for reference)
    """
    
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers=[256, 128, 64],
        dropout_rate=0.2,
        dropout_type='standard',
        activation='relu',
        use_batch_norm=False,
        use_layer_norm=False,
        weight_decay=0.0
    ):
        super(FlexibleNN, self).__init__()
        
        self.weight_decay = weight_decay  # Store for optimizer configuration
        
        # Build activation function
        activation_fn = self._get_activation(activation)
        
        # Build dropout layer
        dropout_layer = self._get_dropout(dropout_type, dropout_rate)
        
        # Build the network layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_layers:
            # Linear layer
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            # Normalization (batch norm or layer norm, not both)
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif use_layer_norm:
                layers.append(nn.LayerNorm(hidden_dim))
            
            # Activation
            layers.append(activation_fn())
            
            # Dropout
            if dropout_layer is not None:
                layers.append(dropout_layer(dropout_rate))
            
            prev_dim = hidden_dim
        
        # Output layer (no activation, dropout, or normalization)
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def _get_activation(self, activation):
        activations = {
            'relu': nn.ReLU,
            'leaky_relu': lambda: nn.LeakyReLU(0.1),
            'elu': nn.ELU,
            'selu': nn.SELU,
            'gelu': nn.GELU,
            'tanh': nn.Tanh,
            'sigmoid': nn.Sigmoid
        }
        if activation not in activations:
            raise ValueError(f"Unknown activation: {activation}. Choose from {list(activations.keys())}")
        return activations[activation]
    
    def _get_dropout(self, dropout_type, dropout_rate):
        if dropout_type == 'none' or dropout_rate == 0:
            return None
        elif dropout_type == 'standard':
            return nn.Dropout
        elif dropout_type == 'alpha':
            return nn.AlphaDropout  # For use with SELU activation
        else:
            raise ValueError(f"Unknown dropout type: {dropout_type}. Choose from ['standard', 'alpha', 'none']")
    
    def forward(self, x):
        return self.network(x)
    
    def get_optimizer(self, optimizer_type='adam', lr=1e-3):
        """
        Get an optimizer with the configured weight decay (L2 regularization).
        """
        optimizers = {
            'adam': lambda: torch.optim.Adam(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'adamw': lambda: torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'sgd': lambda: torch.optim.SGD(self.parameters(), lr=lr, weight_decay=self.weight_decay, momentum=0.9),
            'rmsprop': lambda: torch.optim.RMSprop(self.parameters(), lr=lr, weight_decay=self.weight_decay)
        }
        if optimizer_type not in optimizers:
            raise ValueError(f"Unknown optimizer: {optimizer_type}. Choose from {list(optimizers.keys())}")
        return optimizers[optimizer_type]()


def train_epoch(model, train_loader, optimizer, criterion, device):
    """
    Train the model for one epoch.
    
    Returns:
    --------
    float : Average training loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def evaluate_epoch(model, val_loader, criterion, device):
    """
    Evaluate the model on validation data.
    
    Returns:
    --------
    tuple : (average loss, predictions, targets)
    """
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            batch_size = X_batch.size(0)
            
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            
            # Weight loss by batch size for correct averaging
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            
            all_predictions.append(predictions.cpu())
            all_targets.append(y_batch.cpu())
    
    # Weighted average loss (accounts for different batch sizes)
    avg_loss = total_loss / total_samples
    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    return avg_loss, all_predictions, all_targets


def train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=32,
    optimizer_type='adam',
    lr=1e-3,
    criterion=None,
    early_stopping_patience=None,
    verbose=True,
    device=None
):
    """
    Train the model for a specified number of epochs.
    
    Parameters:
    -----------
    model : FlexibleNN
        The neural network model
    X_train, y_train : array-like
        Training data
    X_val, y_val : array-like
        Validation data
    epochs : int
        Number of training epochs
    batch_size : int
        Batch size for training
    optimizer_type : str
        Type of optimizer ('adam', 'adamw', 'sgd', 'rmsprop')
    lr : float
        Learning rate
    criterion : nn.Module
        Loss function (default: MSELoss)
    early_stopping_patience : int or None
        Stop training if val loss doesn't improve for this many epochs
    verbose : bool
        Whether to print progress
    device : str or None
        Device to train on ('cuda', 'mps', 'cpu', or None for auto-detect)
    
    Returns:
    --------
    dict : Training history with train_losses, val_losses, best_epoch
    """
    # Auto-detect device
    if device is None:
        if torch.cuda.is_available():
            device = 'cuda'
        elif torch.backends.mps.is_available():
            device = 'mps'
        else:
            device = 'cpu'
    
    device = torch.device(device)
    model = model.to(device)
    
    # Default criterion
    if criterion is None:
        criterion = nn.MSELoss()
    
    # Convert data to tensors
    X_train_t = torch.FloatTensor(X_train.values if hasattr(X_train, 'values') else X_train)
    y_train_t = torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train)
    X_val_t = torch.FloatTensor(X_val.values if hasattr(X_val, 'values') else X_val)
    y_val_t = torch.FloatTensor(y_val.values if hasattr(y_val, 'values') else y_val)
    
    # Create data loaders
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Get optimizer
    optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
    
    # Training history
    history = {
        'train_losses': [],
        'val_losses': [],
        'best_epoch': 0,
        'best_val_loss': float('inf')
    }
    
    # Early stopping
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Evaluate
        val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
        
        history['train_losses'].append(train_loss)
        history['val_losses'].append(val_loss)
        
        # Track best model
        if val_loss < history['best_val_loss']:
            history['best_val_loss'] = val_loss
            history['best_epoch'] = epoch
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Verbose output
        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if early_stopping_patience and patience_counter >= early_stopping_patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}. Best epoch: {history['best_epoch']+1}")
            break
    
    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history


# Example hyperparameter search space (for use with Optuna, Ray Tune, etc.)
HYPERPARAM_SPACE = {
    'hidden_layers': [[128, 64], [256, 128], [256, 128, 64], [512, 256, 128], [512, 256, 128, 64]],
    'dropout_rate': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    'dropout_type': ['standard', 'alpha', 'none'],
    'activation': ['relu', 'leaky_relu', 'elu', 'selu', 'gelu'],
    'use_batch_norm': [True, False],
    'use_layer_norm': [True, False],
    'weight_decay': [0.0, 1e-5, 1e-4, 1e-3, 1e-2],
    'optimizer_type': ['adam', 'adamw', 'sgd'],
    'lr': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    'batch_size': [16, 32, 64, 128]
}

print("FlexibleNN and training functions defined.")

FlexibleNN and training functions defined.


In [3]:
def asseble_y(raw_calphed, temp_range):
    #define the prediction temperature range 
    #possible [0, 50, 100, ..., 2500] in steps of 50
    temperature_range = temp_range  # modify this list to include desired temperatures

    y_raw = raw_calphed.drop(labels=['alloy_string'], axis=1)
    
    # Filter columns to only include those matching the selected temperatures
    # Column format: DF_<PHASE>_T<TEMP>C (e.g., DF_AG2CA_T0C, DF_AG2CA_T1000C)
    temp_pattern = re.compile(r'_T(\d+)C$')
    
    selected_cols = []
    for col in y_raw.columns:
        match = temp_pattern.search(col)
        if match:
            temp = int(match.group(1))
            if temp in temperature_range:
                selected_cols.append(col)
    
    y_filtered = y_raw[selected_cols]
    
    print(f"Selected temperatures: {temperature_range}")
    print(f"Filtered columns: {len(selected_cols)} (from {len(y_raw.columns)} total)")
    print(f"y_filtered shape: {y_filtered.shape}")
    
    return y_filtered


In [4]:
calphed_data = pd.read_csv(r'Data/calphad_alloys_train_opt.csv')


In [5]:
calphed_data.head()

,alloy_string,DF_AG2CA_T0C,DF_AG2CA_T1000C,DF_AG2CA_T100C,DF_AG2CA_T1050C,DF_AG2CA_T1100C,DF_AG2CA_T1150C,DF_AG2CA_T1200C,DF_AG2CA_T1250C,DF_AG2CA_T1300C,...,NF_ZRSI_T50C,NF_ZRSI_T550C,NF_ZRSI_T600C,NF_ZRSI_T650C,NF_ZRSI_T700C,NF_ZRSI_T750C,NF_ZRSI_T800C,NF_ZRSI_T850C,NF_ZRSI_T900C,NF_ZRSI_T950C
0,B22.00Co4.00Fe68.00Y6.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Al10.00Ce60.00Cu20.00Ni10.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
fraction_df = calphed_data.filter(like='NF_')
fraction_df.head()

,NF_AG2CA_T0C,NF_AG2CA_T1000C,NF_AG2CA_T100C,NF_AG2CA_T1050C,NF_AG2CA_T1100C,NF_AG2CA_T1150C,NF_AG2CA_T1200C,NF_AG2CA_T1250C,NF_AG2CA_T1300C,NF_AG2CA_T1350C,...,NF_ZRSI_T50C,NF_ZRSI_T550C,NF_ZRSI_T600C,NF_ZRSI_T650C,NF_ZRSI_T700C,NF_ZRSI_T750C,NF_ZRSI_T800C,NF_ZRSI_T850C,NF_ZRSI_T900C,NF_ZRSI_T950C
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Average of all non-zero values in entire dataframe
avg_nonzero = fraction_df.replace(0, np.nan).mean().mean()

# Average per column (non-zero only)
avg_per_col = fraction_df.replace(0, np.nan).mean()

# Or using a mask
avg_nonzero = fraction_df[fraction_df != 0].mean().mean()

In [8]:
avg_nonzero

np.float64(0.21158235281429222)

In [9]:
# Extract alloy strings and create formula dataframe
formula_df = pd.DataFrame({'formula': calphed_data['alloy_string']})
print(f"Formula dataframe shape: {formula_df.shape}")
formula_df.head()

Formula dataframe shape: (882, 1)


,formula
0,B22.00Co4.00Fe68.00Y6.00
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00
4,Al10.00Ce60.00Cu20.00Ni10.00


In [10]:
# Generate Composition-Based Feature Vectors using CBFV
# CBFV expects 'formula' column and optionally a 'target' column
# Add a dummy target for featurization (we'll drop it after)
formula_df['target'] = 0

# Generate CBFVs using the magpie element property database
X, y, formulae, skipped = composition.generate_features(formula_df, elem_prop='magpie')
print(f"CBFV features shape: {X.shape}")
print(f"Number of skipped formulas: {len(skipped)}")
X.head()

Processing Input Data: 100%|██████████| 882/882 [00:00<00:00, 28611.24it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 882/882 [00:00<00:00, 16901.39it/s]

	Creating Pandas Objects...
CBFV features shape: (882, 132)
Number of skipped formulas: 0


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,mode_NValence,mode_NsUnfilled,mode_NpUnfilled,mode_NdUnfilled,mode_NfUnfilled,mode_NUnfilled,mode_GSvolume_pa,mode_GSbandgap,mode_GSmagmom,mode_SpaceGroupNumber
0,22.200000,56.280000,48.044699,1926.70000,8.840000,3.620000,124.680000,1.841600,2.000000,0.220000,...,8.0,0.0,0.0,4.0,0.0,4.0,10.730,0.0,2.110663,229.0
1,27.780000,55.100000,60.858759,1736.29190,7.980000,4.100000,143.910000,1.749000,1.420000,0.020000,...,11.0,1.0,0.0,0.0,0.0,1.0,11.070,0.0,0.000000,225.0
2,24.359036,58.662966,53.217519,2293.90019,8.946795,3.734973,123.841484,1.995602,1.859986,0.360036,...,8.0,0.0,0.0,4.0,0.0,4.0,10.730,0.0,2.110663,229.0
3,29.000000,53.040000,65.476074,1923.65240,4.760000,4.080000,146.560000,1.517600,1.800000,0.000000,...,4.0,0.0,0.0,8.0,0.0,8.0,23.195,0.0,0.000000,194.0
4,44.700000,35.200000,105.346294,1180.30100,6.300000,5.100000,173.300000,1.404000,1.800000,0.100000,...,4.0,0.0,0.0,9.0,13.0,22.0,37.240,0.0,0.000000,194.0


In [14]:
# Create 5 CV groups by clustering similar alloys together
# Standardize features for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Use KMeans to group similar alloys into 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cv_groups = kmeans.fit_predict(X_scaled)

# Add group assignments to a dataframe for reference
cv_group_df = pd.DataFrame({
    'formula': formulae,
    'cv_group': cv_groups
})

print("CV Group distribution:")
print(cv_group_df['cv_group'].value_counts().sort_index())
print(f"\nTotal samples: {len(cv_groups)}")
cv_group_df.head(10)

CV Group distribution:
cv_group
0    200
1    314
2    149
3    125
4     94
Name: count, dtype: int64

Total samples: 882


,formula,cv_group
0,B22.00Co4.00Fe68.00Y6.00,0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,1
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,0
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,3
4,Al10.00Ce60.00Cu20.00Ni10.00,2
5,Cu20.00Gd10.00Mg65.00Ni5.00,3
6,Ca55.00Cu20.00Mg25.00,4
7,Ag5.00Al12.50Cu15.00Fe5.00La62.50,2
8,B5.00C10.00Co35.00Fe40.00P10.00,0
9,Ca55.00Mg20.00Zn25.00,4


In [15]:
# Check column naming convention to distinguish phase fractions from driving forces
sample_cols = calphed_data.columns[:50].tolist()
print("Sample column names:")
for col in sample_cols:
    print(f"  {col}")

# Check for prefixes
print("\nUnique prefixes (first part before _):")
prefixes = set()
for col in calphed_data.columns:
    if '_' in col:
        prefix = col.split('_')[0]
        prefixes.add(prefix)
print(prefixes)

Sample column names:
  alloy_string
  DF_AG2CA_T0C
  DF_AG2CA_T1000C
  DF_AG2CA_T100C
  DF_AG2CA_T1050C
  DF_AG2CA_T1100C
  DF_AG2CA_T1150C
  DF_AG2CA_T1200C
  DF_AG2CA_T1250C
  DF_AG2CA_T1300C
  DF_AG2CA_T1350C
  DF_AG2CA_T1400C
  DF_AG2CA_T1450C
  DF_AG2CA_T1500C
  DF_AG2CA_T150C
  DF_AG2CA_T1550C
  DF_AG2CA_T1600C
  DF_AG2CA_T1650C
  DF_AG2CA_T1700C
  DF_AG2CA_T1750C
  DF_AG2CA_T1800C
  DF_AG2CA_T1850C
  DF_AG2CA_T1900C
  DF_AG2CA_T1950C
  DF_AG2CA_T2000C
  DF_AG2CA_T200C
  DF_AG2CA_T2050C
  DF_AG2CA_T2100C
  DF_AG2CA_T2150C
  DF_AG2CA_T2200C
  DF_AG2CA_T2250C
  DF_AG2CA_T2300C
  DF_AG2CA_T2350C
  DF_AG2CA_T2400C
  DF_AG2CA_T2450C
  DF_AG2CA_T2500C
  DF_AG2CA_T250C
  DF_AG2CA_T300C
  DF_AG2CA_T350C
  DF_AG2CA_T400C
  DF_AG2CA_T450C
  DF_AG2CA_T500C
  DF_AG2CA_T50C
  DF_AG2CA_T550C
  DF_AG2CA_T600C
  DF_AG2CA_T650C
  DF_AG2CA_T700C
  DF_AG2CA_T750C
  DF_AG2CA_T800C
  DF_AG2CA_T850C

Unique prefixes (first part before _):
{'alloy', 'DF', 'NF'}


In [16]:
def evaluate_parameters_nn(parameters, batch_size=32, epochs=1000, verbose=True):
    
    # Extract hyperparameters from parameters dict with defaults
    batch_size = parameters.get('batch_size', batch_size)
    hidden_layers = parameters.get('hidden_layers', [256, 128, 64])
    dropout_rate = parameters.get('dropout_rate', 0.2)
    dropout_type = parameters.get('dropout_type', 'standard')
    activation = parameters.get('activation', 'relu')
    use_batch_norm = parameters.get('use_batch_norm', False)
    use_layer_norm = parameters.get('use_layer_norm', False)
    weight_decay = parameters.get('weight_decay', 0.0)
    optimizer_type = parameters.get('optimizer_type', 'adam')
    lr = parameters.get('lr', 1e-3)
    early_stopping_patience = parameters.get('early_stopping_patience', 10)

    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")

    # Assemble y data, the function filters for desired temperatures
    temp_range = list(range(500,1500,50))  # modify this list to include desired temperatures
    y_data = asseble_y(calphed_data, temp_range)
    x_data = X.copy()

    # Get input and output dimensions
    input_dim = x_data.shape[1]
    output_dim = y_data.shape[1]
    print(f"Input dim: {input_dim}, Output dim: {output_dim}")

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):  # 0-4 for 5 groups from KMeans
        print(f"\n{'='*50}")
        print(f"Fold {test_group}")
        print('='*50)
        
        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data in train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]

        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]
        
        # Create train and validation data
        X_train, X_val, y_train, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Scale the x data
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

        #scale the y data
        y_scaler = StandardScaler()
        y_train_scaled = pd.DataFrame(y_scaler.fit_transform(y_train), columns=y_train.columns, index=y_train.index)
        y_val_scaled = pd.DataFrame(y_scaler.transform(y_val), columns=y_val.columns, index=y_val.index)
        y_test_scaled = pd.DataFrame(y_scaler.transform(y_test), columns=y_test.columns, index=y_test.index)

        # Convert DataFrames to PyTorch tensors
        X_train_t = torch.FloatTensor(X_train_scaled.values)
        y_train_t = torch.FloatTensor(y_train_scaled.values)
        X_val_t = torch.FloatTensor(X_val_scaled.values)
        y_val_t = torch.FloatTensor(y_val_scaled.values)
        X_test_t = torch.FloatTensor(x_test_scaled.values)
        y_test_t = torch.FloatTensor(y_test_scaled.values)

        # Create TensorDatasets
        train_dataset = TensorDataset(X_train_t, y_train_t)
        val_dataset = TensorDataset(X_val_t, y_val_t)
        test_dataset = TensorDataset(X_test_t, y_test_t)

        # Create DataLoaders
        # drop_last=True for train_loader to avoid batch size of 1 (causes BatchNorm to fail)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # Create the model
        model = FlexibleNN(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            dropout_type=dropout_type,
            activation=activation,
            use_batch_norm=use_batch_norm,
            use_layer_norm=use_layer_norm,
            weight_decay=weight_decay
        ).to(device)

        # Get optimizer and criterion
        optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
        criterion = nn.MSELoss()

        # Training loop
        best_val_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # Train
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            
            # Evaluate on validation
            val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            # Track best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Verbose output
            if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
            
            # Early stopping
            if early_stopping_patience and patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Restore best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            model.to(device)

        # Evaluate on test set (scaled)
        test_loss_scaled, test_preds_scaled, test_targets_scaled = evaluate_epoch(model, test_loader, criterion, device)
        
        # Inverse transform predictions and targets to original scale
        test_preds_original = y_scaler.inverse_transform(test_preds_scaled.numpy())
        test_targets_original = y_scaler.inverse_transform(test_targets_scaled.numpy())
        
        # Get column names for separating DF (Driving Force) vs NF (Phase Fraction)
        col_names = y_data.columns.tolist()
        df_cols = [i for i, col in enumerate(col_names) if col.startswith('DF_')]
        nf_cols = [i for i, col in enumerate(col_names) if col.startswith('NF_')]
        
        # Calculate overall RMSE and MAE on original scale (all values)
        mse_original = np.mean((test_preds_original - test_targets_original) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds_original - test_targets_original))
        
        # Calculate metrics for non-zero targets only
        # Use a small threshold to handle floating point precision
        nonzero_threshold = 1e-6
        
        # Overall non-zero metrics
        nonzero_mask = np.abs(test_targets_original) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds_original[nonzero_mask] - test_targets_original[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Driving Force columns (DF_) - non-zero only
        if df_cols:
            df_preds = test_preds_original[:, df_cols]
            df_targets = test_targets_original[:, df_cols]
            df_nonzero_mask = np.abs(df_targets) > nonzero_threshold
            if np.any(df_nonzero_mask):
                df_errors = df_preds[df_nonzero_mask] - df_targets[df_nonzero_mask]
                df_rmse = np.sqrt(np.mean(df_errors ** 2))
                df_mae = np.mean(np.abs(df_errors))
                n_df_nonzero = np.sum(df_nonzero_mask)
            else:
                df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        else:
            df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Phase Fraction columns (NF_) - non-zero only
        if nf_cols:
            nf_preds = test_preds_original[:, nf_cols]
            nf_targets = test_targets_original[:, nf_cols]
            nf_nonzero_mask = np.abs(nf_targets) > nonzero_threshold
            if np.any(nf_nonzero_mask):
                nf_errors = nf_preds[nf_nonzero_mask] - nf_targets[nf_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(nf_errors ** 2))
                nf_mae = np.mean(np.abs(nf_errors))
                n_nf_nonzero = np.sum(nf_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        
        print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
        print(f"Driving Force (DF) - RMSE: {df_rmse:.4f}, MAE: {df_mae:.4f}  [{n_df_nonzero:,} non-zero values]")
        print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'best_val_loss': best_val_loss,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'df_rmse': df_rmse,
            'df_mae': df_mae,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae,
            'train_losses': train_losses,
            'val_losses': val_losses
        })
        
        all_test_predictions.append(torch.FloatTensor(test_preds_original))
        all_test_targets.append(torch.FloatTensor(test_targets_original))

    # Summary across all folds
    print(f"\n{'='*50}")
    print("Cross-Validation Summary (non-zero targets only)")
    print('='*50)
    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])
    
    # Separate metrics for DF and NF (already non-zero)
    avg_df_rmse = np.mean([r['df_rmse'] for r in fold_results])
    avg_df_mae = np.mean([r['df_mae'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    
    print(f"Overall (all values):")
    print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
    print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
    print(f"\nOverall (non-zero only):")
    print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
    print(f"\nDriving Force (DF, non-zero only):")
    print(f"  Avg RMSE: {avg_df_rmse:.4f}, MAE: {avg_df_mae:.4f}")
    print(f"\nPhase Fraction (NF, non-zero only):")
    print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_df_rmse': avg_df_rmse,
        'avg_df_mae': avg_df_mae,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets
    }

In [18]:
parameters = {
    'hidden_layers': [1024, 2048],
    'use_batch_norm': True,
    'weight_decay': 1e-4,
    'lr': 1e-3,
    'early_stopping_patience': 15
}
results = evaluate_parameters_nn(parameters)

Using device: mps
Selected temperatures: [500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200, 1250, 1300, 1350, 1400, 1450]
Filtered columns: 32160 (from 82008 total)
y_filtered shape: (882, 32160)
Input dim: 132, Output dim: 32160

Fold 0
Train: 545, Val: 137, Test: 200
Epoch 1/1000 - Train Loss: 0.476638 - Val Loss: 194.672120
Epoch 21/1000 - Train Loss: 0.150280 - Val Loss: 194.524170
Epoch 41/1000 - Train Loss: 0.136618 - Val Loss: 194.518540
Early stopping at epoch 43
Overall (all)      - RMSE: 2.0659, MAE: 0.6774
Overall (non-zero) - RMSE: 5.9038, MAE: 3.7482  [273,992 values]
Driving Force (DF) - RMSE: 6.0267, MAE: 3.8957  [262,887 non-zero values]
Phase Fraction (NF) - RMSE: 0.3497, MAE: 0.2577  [11,105 non-zero values]

Fold 1
Train: 454, Val: 114, Test: 314
Epoch 1/1000 - Train Loss: 0.523370 - Val Loss: 29.478645
Epoch 21/1000 - Train Loss: 0.183208 - Val Loss: 29.336443
Epoch 41/1000 - Train Loss: 0.166161 - Val Loss: 29.328748
Epoch 61/1000 - Tr

In [19]:

def evaluate_parameters_xgb(parameters, verbose=True):
    """
    Evaluate XGBoost model with given hyperparameters using the same 5-fold CV setup.
    
    Parameters:
    -----------
    parameters : dict
        XGBoost hyperparameters:
        - n_estimators: Number of boosting rounds (default: 100)
        - max_depth: Maximum tree depth (default: 6)
        - learning_rate: Boosting learning rate (default: 0.1)
        - subsample: Subsample ratio of training instances (default: 1.0)
        - colsample_bytree: Subsample ratio of columns (default: 1.0)
        - min_child_weight: Minimum sum of instance weight in a child (default: 1)
        - reg_alpha: L1 regularization term (default: 0)
        - reg_lambda: L2 regularization term (default: 1)
        - gamma: Minimum loss reduction for split (default: 0)
        - early_stopping_rounds: Stop if no improvement (default: 10)
    verbose : bool
        Whether to print progress
    
    Returns:
    --------
    dict : Results including fold metrics and averages
    """
    
    # Extract hyperparameters with defaults
    n_estimators = parameters.get('n_estimators', 100)
    max_depth = parameters.get('max_depth', 6)
    learning_rate = parameters.get('learning_rate', 0.1)
    subsample = parameters.get('subsample', 1.0)
    colsample_bytree = parameters.get('colsample_bytree', 1.0)
    min_child_weight = parameters.get('min_child_weight', 1)
    reg_alpha = parameters.get('reg_alpha', 0)
    reg_lambda = parameters.get('reg_lambda', 1)
    gamma = parameters.get('gamma', 0)
    early_stopping_rounds = parameters.get('early_stopping_rounds', 10)
    n_jobs = parameters.get('n_jobs', -1)
    
    # Assemble y data
    temp_range = list(range(500, 1500, 50))
    y_data = asseble_y(calphed_data, temp_range)
    x_data = X.copy()
    
    print(f"Input dim: {x_data.shape[1]}, Output dim: {y_data.shape[1]}")
    
    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []
    
    for test_group in range(5):
        print(f"\n{'='*50}")
        print(f"Fold {test_group}")
        print('='*50)
        
        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group
        
        # Split the x and y data
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]
        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]
        
        # Create train and validation split
        X_train, X_val, y_train_split, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )
        
        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")
        
        # Scale the x data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(x_test)
        
        # Note: XGBoost doesn't require y scaling, but we keep it for consistency
        # and to match the NN evaluation setup
        y_scaler = StandardScaler()
        y_train_scaled = y_scaler.fit_transform(y_train_split)
        y_val_scaled = y_scaler.transform(y_val)
        y_test_scaled = y_scaler.transform(y_test)
        
        # Create XGBoost model with MultiOutputRegressor wrapper
        base_model = XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            gamma=gamma,
            n_jobs=n_jobs,
            random_state=42,
            verbosity=0,
            early_stopping_rounds=early_stopping_rounds,
            eval_metric='rmse'
        )
        
        model = MultiOutputRegressor(base_model, n_jobs=1)
        
        # Fit with early stopping using validation set
        # MultiOutputRegressor doesn't directly support eval_set, so we need a workaround
        # We'll fit each estimator individually with early stopping
        if verbose:
            print("Training XGBoost model...")
        
        # Train the multi-output model
        # For early stopping with MultiOutputRegressor, we need to fit manually
        model.estimators_ = []
        best_iterations = []
        
        for i in range(y_train_scaled.shape[1]):
            estimator = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                gamma=gamma,
                n_jobs=n_jobs,
                random_state=42,
                verbosity=0,
                early_stopping_rounds=early_stopping_rounds,
                eval_metric='rmse'
            )
            estimator.fit(
                X_train_scaled, y_train_scaled[:, i],
                eval_set=[(X_val_scaled, y_val_scaled[:, i])],
                verbose=False
            )
            model.estimators_.append(estimator)
            best_iterations.append(estimator.best_iteration)
        
        avg_best_iter = np.mean(best_iterations)
        if verbose:
            print(f"Average best iteration: {avg_best_iter:.1f}")
        
        # Predict on test set (scaled)
        test_preds_scaled = np.column_stack([
            est.predict(X_test_scaled) for est in model.estimators_
        ])
        
        # Inverse transform predictions and targets to original scale
        test_preds_original = y_scaler.inverse_transform(test_preds_scaled)
        test_targets_original = y_test.values
        
        # Get column names for separating DF vs NF
        col_names = y_data.columns.tolist()
        df_cols = [i for i, col in enumerate(col_names) if col.startswith('DF_')]
        nf_cols = [i for i, col in enumerate(col_names) if col.startswith('NF_')]
        
        # Calculate overall RMSE and MAE on original scale
        mse_original = np.mean((test_preds_original - test_targets_original) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds_original - test_targets_original))
        
        # Calculate metrics for non-zero targets only
        nonzero_threshold = 1e-6
        
        # Overall non-zero metrics
        nonzero_mask = np.abs(test_targets_original) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds_original[nonzero_mask] - test_targets_original[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0
        
        # Driving Force columns (DF_) - non-zero only
        if df_cols:
            df_preds = test_preds_original[:, df_cols]
            df_targets = test_targets_original[:, df_cols]
            df_nonzero_mask = np.abs(df_targets) > nonzero_threshold
            if np.any(df_nonzero_mask):
                df_errors = df_preds[df_nonzero_mask] - df_targets[df_nonzero_mask]
                df_rmse = np.sqrt(np.mean(df_errors ** 2))
                df_mae = np.mean(np.abs(df_errors))
                n_df_nonzero = np.sum(df_nonzero_mask)
            else:
                df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        else:
            df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        
        # Phase Fraction columns (NF_) - non-zero only
        if nf_cols:
            nf_preds = test_preds_original[:, nf_cols]
            nf_targets = test_targets_original[:, nf_cols]
            nf_nonzero_mask = np.abs(nf_targets) > nonzero_threshold
            if np.any(nf_nonzero_mask):
                nf_errors = nf_preds[nf_nonzero_mask] - nf_targets[nf_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(nf_errors ** 2))
                nf_mae = np.mean(np.abs(nf_errors))
                n_nf_nonzero = np.sum(nf_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        
        print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
        print(f"Driving Force (DF) - RMSE: {df_rmse:.4f}, MAE: {df_mae:.4f}  [{n_df_nonzero:,} non-zero values]")
        print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")
        
        # Store fold results
        fold_results.append({
            'fold': test_group,
            'avg_best_iteration': avg_best_iter,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'df_rmse': df_rmse,
            'df_mae': df_mae,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae
        })
        
        all_test_predictions.append(test_preds_original)
        all_test_targets.append(test_targets_original)
    
    # Summary across all folds
    print(f"\n{'='*50}")
    print("Cross-Validation Summary (XGBoost)")
    print('='*50)
    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])
    
    avg_df_rmse = np.mean([r['df_rmse'] for r in fold_results])
    avg_df_mae = np.mean([r['df_mae'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    
    print(f"Overall (all values):")
    print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
    print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
    print(f"\nOverall (non-zero only):")
    print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
    print(f"\nDriving Force (DF, non-zero only):")
    print(f"  Avg RMSE: {avg_df_rmse:.4f}, MAE: {avg_df_mae:.4f}")
    print(f"\nPhase Fraction (NF, non-zero only):")
    print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")
    
    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_df_rmse': avg_df_rmse,
        'avg_df_mae': avg_df_mae,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets
    }


# Example usage:
# xgb_params = {
#     'n_estimators': 500,
#     'max_depth': 8,
#     'learning_rate': 0.05,
#     'subsample': 0.8,
#     'colsample_bytree': 0.8,
#     'early_stopping_rounds': 20
# }
# xgb_results = evaluate_parameters_xgb(xgb_params)

In [20]:
xgb_params = {
     'n_estimators': 500,
     'max_depth': 8,
     'learning_rate': 0.05,
     'subsample': 0.8,
     'colsample_bytree': 0.8,
     'early_stopping_rounds': 20
}
xgb_results = evaluate_parameters_xgb(xgb_params)

Selected temperatures: [500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200, 1250, 1300, 1350, 1400, 1450]
Filtered columns: 32160 (from 82008 total)
y_filtered shape: (882, 32160)
Input dim: 132, Output dim: 32160

Fold 0
Train: 545, Val: 137, Test: 200
Training XGBoost model...


KeyboardInterrupt: 